In [1]:
import sys 
import numpy 
import matplotlib
import sklearn 
print(sys.version)             # 打印 python 版本号
print(numpy.__version__)        # 打印 numpy 包 版本号
print(matplotlib.__version__)   # 打印 matplotlib 包 版本号
print(sklearn.__version__)         # 打印 h5py 包 版本号

3.8.19 (default, Mar 20 2024, 19:55:45) [MSC v.1916 64 bit (AMD64)]
1.24.3
3.7.1
1.3.0


In [3]:
# Package imports
import numpy as np
import matplotlib.pyplot as plt
from testCases import *
import sklearn
import sklearn.datasets
import sklearn.linear_model
from planar_utils import plot_decision_boundary, sigmoid, load_planar_dataset, load_extra_datasets

%matplotlib inline

# 设置一个固定的随机种子，以保证接下来的步骤中我们的结果是一致的。
np.random.seed(1) 

In [4]:
# 导入数据
X, Y = load_planar_dataset() 

- 定义神经网络结构

In [13]:
# 定义神经网络结构
def layer_sizes(X,Y):
    n_x=X.shape # n_x输入层的大小
    n_h=4 #隐藏层的大小
    n_y=Y.shape[0] #输出层大小
    return (n_x,n_h,n_y)

- 初始化随机参数

In [32]:
def initialize_parameters(n_x,n_h,n_y):
    #设置随机种子 
    np.random.seed(2)
    W1 = np.random.randn(n_h,n_x) * 0.01 #返回样本具有标准正态分布
    b1=np.zeros((n_h,1)) #注意维度
    W2=np.random.randn((n_y,n_h))*0.01
    b2=np.zeros((n_y,1))
    
    # 使用断言确保我的数据格式是正确的
    assert(W1.shape == ( n_h , n_x ))
    assert(b1.shape == ( n_h , 1 ))
    assert(W2.shape == ( n_y , n_h ))
    assert(b2.shape == ( n_y , 1 ))
    parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2} #创建字典
    return parameters #调用这个函数将返回包含这些参数的字典

- 前向传播
    -  反向传播所需的值存储在cache中，cache将作为反向传播函数的输入。

In [28]:
def forward_propagation(X, parameters):
    # 从字典 “parameters” 中检索每个参数
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    
    #前向传播
    Z1=np.dot(W1,X)+b1
    A1=np.tanh(Z1)
    Z2=np.dot(W2,A1)+b2
    A2=sigmoid(Z2) #二分类 所以最后一层使用sigmoid
    
    cache = {"Z1": Z1,
             "A1": A1,
             "Z2": Z2,
             "A2": A2}
    
    return A2, cache


- 计算成本 
    - 使用交叉熵损失函数


In [18]:
def compute_cost(A2, Y, parameters):
    #样本数量
    m=Y.shape[1]
    
    #计算交叉熵代价
    logprobs=Y*np.log(A2)+(1-Y)*np.log(1-A2)
    cost=-1/m*np.sum(logprobs)
    # 确保损失是我们期望的维度
    # 例如，turns [[17]] into 17 
    cost = np.squeeze(cost)
    
    return cost

- 后向传播
    - 首先计算梯度

In [41]:
def backward_propagation(parameters, cache, X, Y):

    """
    这个函数就是计算梯度的
    """
    
    m = X.shape[1]
    
    # 首先，从字典“parameters”中检索W1和W2。
    W1 = parameters["W1"]
    W2 = parameters["W2"]
        
    # 还可以从字典“cache”中检索A1和A2。
    A1 = cache["A1"]
    A2 = cache["A2"]
    
    # 反向传播:计算 dW1、db1、dW2、db2。 直接看公式 其他也不用管
    dZ2= A2 - Y
    dW2 = 1 / m * np.dot(dZ2,A1.T)
    db2 = 1 / m * np.sum(dZ2,axis=1,keepdims=True)
    dZ1 = np.dot(W2.T,dZ2) * (1-np.power(A1,2))
    dW1 = 1 / m * np.dot(dZ1,X.T)
    db1 = 1 / m * np.sum(dZ1,axis=1,keepdims=True)
    
    grads = {"dW1": dW1,
             "db1": db1,
             "dW2": dW2,
             "db2": db2}
    
    return grads 

   - 接着利用梯度更新参数

In [42]:
def update_parameters(parameters, grads, lr = 1.2):
    # 从字典“parameters”中检索每个参数
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    
    # 从字典“梯度”中检索每个梯度
    dW1 = grads["dW1"]
    db1 = grads["db1"]
    dW2 = grads["dW2"]
    db2 = grads["db2"]
    
    #开始更新
    W1=W1-lr*dW1
    b1=b1-lr*db1
    W2=W2-lr*dW2
    b2=b2-lr*db2
    
    #返回更新后的参数
    parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2}
    
    return parameters

# 整合到网络中

In [43]:
def nn_model(X,Y,n_h,num_iterations=10000,print_cost=False):
    np.random.seed(3) #随机初始化参数
    #首先获取网络各层的尺寸
    (n_x,n_h ,n_y)= layer_sizes(X, Y)

    #然后初始化参数
    parameters = initialize_parameters(n_x, n_h, n_y)
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    
    #梯度下降 也就是模型的训练过程
    for i in range(num_iterations):
        #前向传播
        A2,cache=forward_propagation(X,parameters)
        #计算成本
        cost=compute_cost(A2,Y,parameters)
        #反向传播
        grads=backward_propagation(parameters,cache,X,Y)
        #更新参数
        parameters=update_parameters(parameters,grads)
        
        # 每1000次迭代打印成本
        if print_cost and i%1000==0:
            print ("Cost after iteration %i: %f" %(i, cost))
    return parameters


- 测试一下

In [44]:
# 测试一下 nn_model 函数
X_assess, Y_assess = nn_model_test_case()
parameters = nn_model(X_assess, Y_assess, 4, num_iterations=10000, print_cost=False)

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

TypeError: 'tuple' object cannot be interpreted as an integer

# 预测
- 利用构建好的模型预测数据

In [35]:
def predict(parameters,X):
    A2,cache=forward_propagation(X,parameters)
    prediction-np.round(A2) #round是对数组或单个数值进行四舍五入。
    return prediction

# 正式运行
- 神经网络的参数在网络定义时被初始化，但参数的最终确定是在训练过程中，通过前向传播、反向传播和梯度更新逐步优化和确定的。
- 训练结束时，参数才确定为最终值，经过训练优化后的参数能够最好地拟合训练数据。

In [36]:
# 首先构建一个n_h维隐藏层的nn网路
X, Y = load_planar_dataset()  #引入数据集
parameters = nn_model(X, Y, n_h = 4, num_iterations = 10000, print_cost=True)
#至此，模型中的参数已经训练完毕

# 预测
predictions = predict(parameters, X)
print ('准确率: %d' % float((np.dot(Y, predictions.T) + np.dot(1 - Y, 1 - predictions.T)) / float(Y.size) * 100) + '%')

# np.dot(Y, predictions.T) 计算 Y 和 predictions 的点积，这个点积表示模型预测正确的正类样本的数量
# np.dot(1 - Y, 1 - predictions.T) 计算 1 - Y 和 1 - predictions.T 的点积，这个点积表示模型预测正确的负类样本的数量。

# 整体上，这个表达式表示预测正确的样本数量占总样本数量的比例，并转化为百分比形式



TypeError: 'tuple' object cannot be interpreted as an integer